# R1 — 5-arm Experiment (1 GPT model)

**Scope**: 1 generator model × 5 arms × 200 BHXH questions = 1000 inferences

**Arms**: `graphrag`, `llm_only`, `elite_no_retrieval`, `elite_ontology`, `elite_graphrag`

**Parallelism**: 5 arms chạy concurrent — T4 16GB host được 4-5 BGE-M3 instances (vs 1 trên RTX 3050 4GB local)

**Wall time on Colab Pro T4**: ~1-2h inference + ~1.5h metrics = ~3h

**Supported metrics** (17 total): citation_validity, citation_recall/precision, faithfulness, content_hallucination_rate, invented_citation_rate, answer_relevance, bertscore_f1, cost_usd, latency_s, prolog_success_rate (+ 3 prolog), pairwise_consensus, api_error_rate, McNemar/Bootstrap/Bonferroni significance.

## Prerequisites
1. Code repo trên GDrive: `/MyDrive/legal-graph-kb`
2. Neo4j Aura instance + secrets: `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`

## ⚙️ CONFIG

In [ ]:
# ═══ MODEL CONFIG ═══
R1_MODEL       = 'gpt-4o-mini'   # generator cho 5 arms. Thử: gpt-4.1, gpt-4o, gpt-5-mini
JUDGE_MODEL    = 'gpt-4o-mini'   # LLM-as-Judge. Rigorous: gpt-4o (full)

# ═══ SCOPE ═══
N_QUESTIONS    = 200             # max 200 (set 10 cho pilot)
BACKFILL_PLAIN = True            # backfill plain_answer cho elite arms (~$0.30)
MAX_PARALLEL   = 5               # 5 arms concurrent

# ═══ ARMS ═══
ARMS           = ['graphrag', 'llm_only',
                   'elite_no_retrieval', 'elite_ontology', 'elite_graphrag']

# ═══ PATHS ═══
REPO_GDRIVE    = '/content/drive/MyDrive/legal-graph-kb'
RESULTS_GDRIVE = '/content/drive/MyDrive/legal-graph-kb-results'

print(f'R1 generator: {R1_MODEL}')
print(f'Judge model:  {JUDGE_MODEL}')
print(f'Arms:         {ARMS}')
print(f'Total inferences: {len(ARMS) * N_QUESTIONS}')

# Phase 1 — Setup

## 1.1 GPU + GDrive + secrets

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

from google.colab import drive, userdata
import os, sys, subprocess, time, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
drive.mount('/content/drive')

for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']:
    os.environ[k] = userdata.get(k) or ''
os.environ['NEO4J_DATABASE'] = 'neo4j'
os.environ['EMBED_DEVICE'] = 'cuda'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print({k: ('set' if os.environ.get(k) else 'MISSING')
       for k in ['OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD']})

## 1.2 Copy repo + install deps + symlink

In [ ]:
REPO_DIR = '/content/legal-graph-kb'
if not os.path.exists(REPO_DIR):
    !cp -r $REPO_GDRIVE $REPO_DIR
%cd $REPO_DIR
sys.path.insert(0, REPO_DIR)

!pip install -q neo4j sentence-transformers openai python-dotenv bert-score tqdm 2>&1 | tail -3
!apt-get install -y -q swi-prolog 2>&1 | tail -2

os.makedirs(f'{RESULTS_GDRIVE}/data/eval', exist_ok=True)
if os.path.islink('data/eval') or not os.path.exists('data/eval'):
    !rm -rf data/eval && ln -s $RESULTS_GDRIVE/data/eval data/eval
!ls -la data/eval | head -3

## 1.3 Neo4j Aura verify + push KG

In [ ]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver(os.environ['NEO4J_URI'],
    auth=(os.environ['NEO4J_USER'], os.environ['NEO4J_PASSWORD']))
with driver.session(database='neo4j') as s:
    n_art = s.run('MATCH (n:Article) RETURN count(n) AS c').single()['c']
    print(f'Aura: {n_art} Articles')
    has_vec = any('clause_vec' in (i.get('name') or '')
                   for i in s.run('SHOW INDEXES').data())
    if not has_vec:
        s.run('''CREATE VECTOR INDEX clause_vec IF NOT EXISTS
                 FOR (c:Clause) ON c.embedding
                 OPTIONS {indexConfig: {`vector.dimensions`: 1024,
                                         `vector.similarity_function`: 'cosine'}}''')
driver.close()
if n_art == 0:
    !python -m src.load_neo4j 2>&1 | tail -10

## 1.4 Smoke test với R1_MODEL

In [ ]:
os.environ['OPENAI_MODEL'] = R1_MODEL
from src.rag_query import RagPipeline
from experiments.elite_pipelines import EliteGraphRAGPipeline

Q = 'Tôi đã đóng BHXH 12 năm, có đủ điều kiện hưởng lương hưu không?'
p = RagPipeline(); _ = p.embed_model
r = p.ask(Q, top_k=8, verbose=False)
print(f'[graphrag × {R1_MODEL}] {r.answer[:180]}')
p.close()

ep = EliteGraphRAGPipeline(model=R1_MODEL)
ans = ep.ask(Q)
print(f'\n[elite × {R1_MODEL}] IRAC[:180]: {ans.answer[:180]}')
print(f'  plain_answer: {ans.plain_answer}')
ep.close()

# Phase 2 — Inference (5 arms parallel)

All 5 arms run concurrent — true parallelism nhờ T4 16GB VRAM.

In [ ]:
os.makedirs('logs', exist_ok=True)

def run_combo(cmd, name, env_overrides=None):
    t0 = time.time()
    env = os.environ.copy()
    if env_overrides: env.update(env_overrides)
    with open(f'logs/{name}.log', 'w', encoding='utf-8') as f:
        proc = subprocess.run(cmd, shell=True, stdout=f,
                              stderr=subprocess.STDOUT, env=env)
    return name, proc.returncode, time.time() - t0

def parallel(cmds_with_env, max_workers):
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(run_combo, c, n, e): n for c, n, e in cmds_with_env}
        for fut in as_completed(futs):
            n, code, dur = fut.result()
            status = '✓' if code == 0 else f'✗ exit={code}'
            print(f'[{time.strftime("%H:%M:%S")}] {status} {n} ({dur:.0f}s)')
    print(f'\nWall time: {(time.time()-t0)/60:.1f} min')

# Build cmd list — 5 R1 arms parallel, mỗi arm dùng R1_MODEL
cmds = [
    (f'python -m experiments.run_inference --arm {arm} --n {N_QUESTIONS}',
     f'r1_{arm}',
     {'OPENAI_MODEL': R1_MODEL})
    for arm in ARMS
]
print(f'Launching {len(cmds)} arms parallel (max {MAX_PARALLEL} concurrent)...\n')
parallel(cmds, max_workers=MAX_PARALLEL)

## 2.x Progress check

In [ ]:
for d in sorted(Path('data/eval/results').iterdir()) if Path('data/eval/results').exists() else []:
    if d.is_dir():
        n = len([f for f in d.glob('A*.json') if not f.name.endswith('.error.json')])
        n_api = sum(1 for f in d.glob('A*.json')
                    if not f.name.endswith('.error.json')
                    and json.loads(f.read_text(encoding='utf-8')).get('prompt_tokens') == 0)
        api_tag = f' ({n_api} api_err)' if n_api else ''
        print(f'  {d.name:<30} {n}/{N_QUESTIONS}{api_tag}')

# Phase 3 — Metrics + Report

## 3.1 Backfill plain_answer cho elite arms (optional)

In [ ]:
if BACKFILL_PLAIN:
    os.environ['BACKFILL_MODEL'] = 'gpt-4o-mini'
    # Chỉ backfill R1 elite arms
    !python -m experiments.rerender_plain_answer \
        --combos elite_no_retrieval,elite_ontology,elite_graphrag 2>&1 | tail -15
else:
    print('Skipped')

## 3.2 Compute R1 metrics (judge = JUDGE_MODEL)

In [ ]:
env_judge = os.environ.copy()
env_judge['OPENAI_MODEL'] = JUDGE_MODEL
subprocess.run('python -m experiments.compute_metrics',
               shell=True, env=env_judge)

## 3.3 Audit fixes + report + significance

In [ ]:
!python -m experiments.audit_apply_fixes 2>&1 | tail -8
!python -m experiments.audit_apply_fixes_v2 2>&1 | tail -8
!python -m experiments.generate_report 2>&1 | tail -3
!python -m experiments.compute_significance 2>&1 | tail -3

# Phase 4 — Visualizations 📈 (R1 only)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

def ch(rec, *keys):
    v = rec
    for k in keys:
        if not isinstance(v, dict): return None
        v = v.get(k)
    return v

r1_data = json.loads(Path('data/eval/metrics.json').read_text(encoding='utf-8'))
rows = []
for arm, recs in r1_data.items():
    for r in recs:
        if r.get('api_error'): continue
        ps = ch(r, 'prolog_rollback', 'prolog_success')
        fs = ch(r, 'prolog_rollback', 'first_try_success')
        rows.append({
            'arm': arm, 'stt': r['stt'],
            'citation_validity': ch(r, 'citation_validity', 'validity_rate'),
            'citation_recall': ch(r, 'citation_recall', 'recall'),
            'citation_precision': ch(r, 'citation_precision', 'precision'),
            'faithfulness': ch(r, 'faithfulness', 'faithfulness'),
            'content_hallu': ch(r, 'hallucination', 'content_hallucination_rate'),
            'invented_cit': ch(r, 'hallucination', 'invented_citation_rate'),
            'answer_relevance': ch(r, 'answer_relevance', 'answer_relevance'),
            'bertscore_f1': ch(r, 'bertscore', 'bertscore_f1'),
            'cost_usd': ch(r, 'cost', 'cost_usd'),
            'latency_s': ch(r, 'latency', 'latency_s'),
            'prolog_success': 1.0 if ps else 0.0 if ps is not None else None,
            'first_try_success': 1.0 if fs else 0.0 if fs is not None else None,
        })
df = pd.DataFrame(rows)
print(f'R1 clean records: {len(df)}')
df.head()

## 4.1 Bar chart — Prolog success per elite arm

In [ ]:
elite = df[df['arm'].str.startswith('elite_') & df['prolog_success'].notna()]
if not elite.empty:
    agg = elite.groupby('arm').agg(
        prolog_success_rate=('prolog_success', 'mean'),
        first_try_rate=('first_try_success', 'mean'),
        n=('prolog_success', 'count'),
    ).reset_index().sort_values('prolog_success_rate', ascending=False)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(agg))
    ax.bar(x - 0.2, agg['prolog_success_rate'], 0.4, label='success_rate', color='#2ca02c')
    ax.bar(x + 0.2, agg['first_try_rate'], 0.4, label='first_try_rate', color='#1f77b4')
    ax.set_xticks(x); ax.set_xticklabels(agg['arm'], rotation=20, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('rate')
    ax.set_title(f'R1 Prolog Reliability ({R1_MODEL}) — API errors excluded')
    ax.legend()
    for i, row in agg.iterrows():
        ax.text(i, row['prolog_success_rate']+0.01, f"n={row['n']}",
                 ha='center', fontsize=8, color='gray')
    plt.tight_layout(); plt.show()

## 4.2 Heatmap — Metric matrix per arm

In [ ]:
METRIC_COLS = ['citation_validity', 'citation_recall', 'citation_precision',
                'faithfulness', 'content_hallu', 'invented_cit',
                'answer_relevance', 'bertscore_f1']
heat = df.groupby('arm')[METRIC_COLS].mean()
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0, vmax=1, cbar_kws={'label': 'mean'}, ax=ax)
ax.set_title(f'R1 Metric Matrix ({R1_MODEL}) — mean per arm')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## 4.3 Scatter — Cost vs Faithfulness

In [ ]:
agg = df.groupby('arm').agg(
    cost_usd=('cost_usd', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    latency_s=('latency_s', 'mean'),
).reset_index().dropna()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(agg['cost_usd'], agg['faithfulness'],
           s=agg['latency_s']*30, alpha=0.6,
           c=range(len(agg)), cmap='tab10')
for _, row in agg.iterrows():
    ax.annotate(row['arm'], (row['cost_usd'], row['faithfulness']),
                fontsize=9, xytext=(5,5), textcoords='offset points')
ax.set_xlabel('Mean cost (USD per question)')
ax.set_ylabel('Mean faithfulness')
ax.set_title(f'R1 Cost vs Faithfulness ({R1_MODEL}) — bubble = latency')
ax.set_xscale('log')
plt.tight_layout(); plt.show()

## 4.4 Box plot — Latency distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df_plot = df.dropna(subset=['latency_s']).copy()
order = df_plot.groupby('arm')['latency_s'].median().sort_values().index.tolist()
sns.boxplot(data=df_plot, x='arm', y='latency_s', order=order, ax=ax)
ax.set_yscale('log')
ax.set_title(f'R1 Latency Distribution ({R1_MODEL}) — log scale')
ax.set_xlabel(''); ax.set_ylabel('seconds')
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

## 4.5 Pairwise consensus — non-baseline arms vs graphrag

In [ ]:
from collections import Counter
rows_pw = []
for arm, recs in r1_data.items():
    if arm == 'graphrag': continue
    for r in recs:
        if r.get('api_error'): continue
        pw = r.get('pairwise_vs_baseline')
        if pw:
            rows_pw.append({'compare': f'{arm} vs graphrag', 'consensus': pw['consensus']})
if rows_pw:
    df_pw = pd.DataFrame(rows_pw)
    pivot = df_pw.groupby(['compare', 'consensus']).size().unstack(fill_value=0)
    pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(10, 4))
    pct.plot(kind='barh', stacked=True, ax=ax, colormap='tab20')
    ax.set_xlabel('% of records')
    ax.set_title(f'R1 Pairwise Consensus ({R1_MODEL}) — non-baseline arms vs graphrag')
    ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

## 4.6 Sortable summary DataFrame

In [ ]:
summary = df.groupby('arm').agg(
    n=('stt', 'count'),
    cit_valid=('citation_validity', 'mean'),
    cit_recall=('citation_recall', 'mean'),
    cit_precision=('citation_precision', 'mean'),
    faithfulness=('faithfulness', 'mean'),
    content_hallu=('content_hallu', 'mean'),
    invented_cit=('invented_cit', 'mean'),
    bertscore=('bertscore_f1', 'mean'),
    cost_usd=('cost_usd', 'mean'),
    latency_s=('latency_s', 'mean'),
    prolog_success=('prolog_success', 'mean'),
).round(4)
summary.style.background_gradient(cmap='RdYlGn', axis=0,
    subset=['cit_valid','cit_recall','cit_precision','faithfulness','bertscore','prolog_success'])\
    .background_gradient(cmap='RdYlGn_r', axis=0,
    subset=['content_hallu','invented_cit','cost_usd','latency_s'])

## 4.7 Display markdown reports + sync + download

In [ ]:
from IPython.display import Markdown, display
for f in ['reports/experiment_report.md', 'reports/significance.md']:
    if os.path.exists(f):
        print(f'\n{"="*70}\n{f}\n{"="*70}')
        display(Markdown(open(f, encoding='utf-8').read()[:5000]))

!mkdir -p $RESULTS_GDRIVE/reports
!rsync -av reports/ $RESULTS_GDRIVE/reports/ 2>&1 | tail -3
!cd /content/legal-graph-kb && zip -rq /content/r1_bundle.zip reports/ data/eval/metrics.json data/eval/metrics.csv
from google.colab import files
files.download('/content/r1_bundle.zip')
print(f'\n✓ R1 pipeline complete ({R1_MODEL})')